# XClinVision: Data Preprocessing & Augmentation
**Day 1: EDA & Data Preparation**

This notebook tests preprocessing pipelines, augmentation strategies,
and validates data transformations for the XClinVision project.

## 1. Setup and Imports

In [ ]:
import os
import sys
from pathlib import Path

sys.path.insert(0, str(Path().absolute().parent / "src"))

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
import torch

from xclinvision.data import ChestXrayDataset, get_train_augmentation, get_val_augmentation

## 2. Load Sample Image

In [ ]:
# Find a sample image
DATA_DIR = Path("../data/raw")

sample_img = None
for dataset_dir in DATA_DIR.iterdir():
    if dataset_dir.is_dir():
        for split in ["train", "val", "test"]:
            split_dir = dataset_dir / split
            if split_dir.exists():
                for class_dir in split_dir.iterdir():
                    if class_dir.is_dir():
                        img_files = list(class_dir.glob("*.jpeg"))
                        if img_files:
                            sample_img = img_files[0]
                            break
                if sample_img:
                    break
        if sample_img:
            break

if sample_img:
    print(f"Sample image: {sample_img}")
    img = Image.open(sample_img)
    img_np = np.array(img)
    print(f"Original shape: {img_np.shape}")
    
    # Display
    plt.figure(figsize=(8, 8))
    plt.imshow(img_np, cmap="gray" if len(img_np.shape) == 2 else None)
    plt.title("Original Image")
    plt.axis("off")
    plt.show()
else:
    print("No sample images found. Please download datasets first.")

## 3. Test Augmentation Pipeline

In [ ]:
def visualize_augmentations(image_np, transform, n_augmentations=5):
    """Visualize augmented versions of an image."""
    
    fig, axes = plt.subplots(1, n_augmentations + 1, figsize=(20, 4))
    
    # Original
    axes[0].imshow(image_np, cmap="gray" if len(image_np.shape) == 2 else None)
    axes[0].set_title("Original")
    axes[0].axis("off")
    
    # Augmentations
    for i in range(n_augmentations):
        augmented = transform(image=image_np)
        aug_img = augmented["image"]
        
        # Convert tensor to numpy if needed
        if torch.is_tensor(aug_img):
            aug_img = aug_img.permute(1, 2, 0).numpy()
        
        axes[i + 1].imshow(aug_img, cmap="gray" if len(aug_img.shape) == 2 else None)
        axes[i + 1].set_title(f"Aug {i+1}")
        axes[i + 1].axis("off")
    
    plt.tight_layout()
    plt.show()

# Test training augmentations
if sample_img:
    img_np = np.array(Image.open(sample_img).convert("RGB"))
    train_transform = get_train_augmentation(img_size=384)
    visualize_augmentations(img_np, train_transform, n_augmentations=5)

## 4. Test Normalization

In [ ]:
def test_normalization(image_np):
    """Test different normalization strategies."""
    
    # ImageNet normalization
    imagenet_transform = A.Compose([
        A.Resize(384, 384),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2()
    ])
    
    # Dataset normalization (to be calculated)
    dataset_transform = A.Compose([
        A.Resize(384, 384),
        A.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5]),
        ToTensorV2()
    ])
    
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    
    # Original
    axes[0].imshow(image_np)
    axes[0].set_title("Original")
    axes[0].axis("off")
    
    # ImageNet normalized
    img_imagenet = imagenet_transform(image=image_np)["image"]
    axes[1].imshow(img_imagenet.permute(1, 2, 0).numpy())
    axes[1].set_title("ImageNet Normalized")
    axes[1].axis("off")
    
    # Dataset normalized
    img_dataset = dataset_transform(image=image_np)["image"]
    axes[2].imshow(img_dataset.permute(1, 2, 0).numpy())
    axes[2].set_title("Dataset Normalized")
    axes[2].axis("off")
    
    plt.tight_layout()
    plt.show()

if sample_img:
    test_normalization(np.array(Image.open(sample_img).convert("RGB")))

## 5. Dataset Class Test

In [ ]:
# Test ChestXrayDataset
DATA_DIR = Path("../data/processed")

if (DATA_DIR / "train").exists():
    dataset = ChestXrayDataset(
        data_dir=DATA_DIR / "train",
        transform=get_train_augmentation(),
    )
    
    print(f"Dataset size: {len(dataset)}")
    
    # Get a sample
    img, label = dataset[0]
    print(f"Sample shape: {img.shape}")
    print(f"Sample label: {label}")
    print(f"Label classes: {dataset.classes}")
else:
    print("Processed data not found. Run preprocessing first.")